In [1]:
# Cell 1: Imports — NO FastFeatureEngineer (EDA uses raw merged data only)
import os
import matplotlib
matplotlib.use('Agg')  # headless rendering
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from retail_iq.preprocessing import load_raw_data, preprocess_dates, merge_datasets
from retail_iq.visualization import plot_ts_decomposition, plot_correlation_heatmap, plot_sales_distribution
from retail_iq.config import OUTPUT_DIR

eda_dir = OUTPUT_DIR / 'eda'
eda_dir.mkdir(parents=True, exist_ok=True)
print('Imports OK. EDA output dir:', eda_dir)

Imports OK. EDA output dir: C:\Users\mypc\Downloads\Retail_Demand_Forecasting\outputs\eda


In [2]:
# Cell 2: Load + Merge (~5s via Parquet)
import time
t = time.time()
train, test, stores, oil, holidays, tx = load_raw_data()
train, test, oil, holidays, tx = preprocess_dates([train, test, oil, holidays, tx])
df = merge_datasets(train, stores, oil, holidays, tx)
print(f'Loaded & merged in {time.time()-t:.2f}s')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.describe()

Loaded & merged in 4.23s
Shape: (3000888, 13)
Columns: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'transactions', 'is_national_holiday']


,id,date,store_nbr,sales,onpromotion,cluster,dcoilwtico,transactions,is_national_holiday
count,3.000888e+06,3000888,3.000888e+06,3.000888e+06,3.000888e+06,3.000888e+06,2.143746e+06,2.778831e+06,3.000888e+06
mean,1.500444e+06,2015-04-24 08:27:04.703000,2.750000e+01,3.577757e+02,2.602770e+00,8.481481e+00,6.790519e+01,1.694030e+03,8.076010e-02
min,0.000000e+00,2013-01-01 00:00:00,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.619000e+01,5.000000e+00,0.000000e+00
25%,7.502218e+05,2014-02-26 18:00:00,1.400000e+01,0.000000e+00,0.000000e+00,4.000000e+00,4.640000e+01,1.043000e+03,0.000000e+00
50%,1.500444e+06,2015-04-24 12:00:00,2.750000e+01,1.100000e+01,0.000000e+00,8.500000e+00,5.338000e+01,1.393000e+03,0.000000e+00
75%,2.250665e+06,2016-06-19 06:00:00,4.100000e+01,1.958473e+02,0.000000e+00,1.300000e+01,9.580000e+01,2.078000e+03,0.000000e+00
max,3.000887e+06,2017-08-15 00:00:00,5.400000e+01,1.247170e+05,7.410000e+02,1.700000e+01,1.106200e+02,8.359000e+03,1.000000e+00
std,8.662819e+05,NaN,1.558579e+01,1.101998e+03,1.221888e+01,4.649735e+00,2.566373e+01,9.640143e+02,2.724664e-01


In [3]:
# Cell 3: Time-Series Decomposition — first 3 store/family combos
samples = df.groupby(['store_nbr', 'family']).size().head(3).index
for store_nbr, family in samples:
    save_path = str(eda_dir / f'ts_decompose_{store_nbr}_{family}.png')
    plot_ts_decomposition(df, store_nbr, family, period=7, save_path=save_path)
    print(f'Saved: {save_path}')

Saved: C:\Users\mypc\Downloads\Retail_Demand_Forecasting\outputs\eda\ts_decompose_1_AUTOMOTIVE.png
Saved: C:\Users\mypc\Downloads\Retail_Demand_Forecasting\outputs\eda\ts_decompose_1_BABY CARE.png
Saved: C:\Users\mypc\Downloads\Retail_Demand_Forecasting\outputs\eda\ts_decompose_1_BEAUTY.png


In [4]:
# Cell 4: Correlation Heatmap (sampled 50K rows — ~24x faster than full 3M)
plot_correlation_heatmap(df, save_path=str(eda_dir / 'correlation_heatmap.png'))
print('Saved: correlation_heatmap.png')

Saved: correlation_heatmap.png


In [5]:
# Cell 5: Sales Distribution (sampled 100K rows)
plot_sales_distribution(df, save_path=str(eda_dir / 'sales_distribution.png'))
print('Saved: sales_distribution.png')

Saved: sales_distribution.png


In [6]:
# Cell 6: Holiday Lift Analysis
# Map holiday type directly from holidays table
holiday_map = holidays.set_index('date')['type']
df['holiday_type'] = df['date'].map(holiday_map).fillna('non_holiday')

holiday_avg = df.groupby('holiday_type')['sales'].mean().reset_index()
plt.figure(figsize=(10, 5))
sns.barplot(x='holiday_type', y='sales', data=holiday_avg)
plt.title('Holiday Lift Analysis — Mean Sales by Holiday Type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(eda_dir / 'holiday_lift.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: holiday_lift.png')

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
# Cell 7: Oil Price vs Aggregate Daily Sales
if 'dcoilwtico' in df.columns:
    daily_sales = df.groupby('date')['sales'].sum().reset_index()
    daily_data = daily_sales.merge(oil[['date', 'dcoilwtico']], on='date', how='left')
    plt.figure(figsize=(12, 6))
    sns.scatterplot(x='dcoilwtico', y='sales', data=daily_data, alpha=0.6)
    plt.title('Oil Price vs Aggregate Daily Sales')
    plt.xlabel('WTI Crude Oil Price (USD)')
    plt.ylabel('Total Daily Sales')
    plt.tight_layout()
    plt.savefig(eda_dir / 'oil_vs_sales.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('Saved: oil_vs_sales.png')
else:
    print('dcoilwtico not in merged df — skipping oil plot')

print('EDA pipeline complete.')

Saved: oil_vs_sales.png
EDA pipeline complete.
